In [17]:
!pip install nltk sastrawi

  Using cached nltk-3.10.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached Sastrawi-1.0.1-py2.py3-none-any.whl.metadata (909 bytes)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached nltk-3.10.3-py3-none-any.whl (1.8 MB)
Using cached Sastrawi-1.0.1-py2.py3-none-any.whl (209 kB)
Using cached click-8.5.0-py3-none-any.whl (125 kB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached tqdm-4.70.1-py3-none-any.whl (80 kB)

   ---------------------------------------- 0/7 [sastrawi]
   ---------------------------------------- 0/7 [sastrawi]
   ---------------------------------------- 0/7 [sastrawi]
   ---------------------------------------- 0/7 [sastrawi]
   ---------------------------------------- 0/7 [sastrawi

In [11]:
import pandas as pd
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Linzz\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Linzz\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Linzz\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
df = pd.read_csv("scrape_kompas_clean.csv")
text_column = "title"  # sesuaikan

In [13]:
def case_folding(text):
    return str(text).lower()

# Contoh
text = "Saya Belajar NLP"
print(case_folding(text))
# Output: saya belajar nlp

saya belajar nlp


In [14]:
def remove_url(text):
    return re.sub(r"http\S+|www\S+", "", text)

# Contoh
text = "Kunjungi https://example.com sekarang"
print(remove_url(text))
# Output: Kunjungi  sekarang

Kunjungi  sekarang


In [15]:
def remove_mention(text):
    return re.sub(r"@\w+", "", text)

# Contoh
text = "@user1 produk ini bagus"
print(remove_mention(text))

 produk ini bagus


In [16]:
def remove_hashtag(text):
    return re.sub(r"#\w+", "", text)

In [17]:
def remove_number(text):
    return re.sub(r"\d+", "", text)

In [18]:
def remove_punctuation(text):
    return text.translate(
        str.maketrans("", "", string.punctuation)
    )

In [19]:
def clean_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

In [20]:
#Gabungkan semua fungsi menjadi satu pipeline:
def clean_text(text):
    text = str(text)
    text = text.lower()           # case folding
    text = remove_url(text)       # hapus URL
    text = remove_mention(text)   # hapus mention
    text = remove_hashtag(text)   # hapus hashtag
    text = remove_number(text)    # hapus angka
    text = remove_punctuation(text)  # hapus punctuation
    text = clean_whitespace(text) # bersihkan whitespace
    return text

#Menerapkan ke dataset:
df["clean_text"] = df[text_column].apply(clean_text)

df[[text_column, "clean_text"]].head(10)

,title,clean_text
0,Deretan Aktor yang Pernah Perankan Spider-Man,deretan aktor yang pernah perankan spiderman
1,Aktor Ryotaro Shimizu Ditemukan Meninggal Duni...,aktor ryotaro shimizu ditemukan meninggal duni...
2,"Aktor Marvelous Mrs. Maisel, Josh Grisetti Men...",aktor marvelous mrs maisel josh grisetti menin...
3,"Aktor Bittersweet Destiny, Wang Kai Ditemukan ...",aktor bittersweet destiny wang kai ditemukan m...
4,"Aktor Thailand, Jorjay Phumhirun Mendadak Meni...",aktor thailand jorjay phumhirun mendadak menin...


In [21]:
stop_words = set(stopwords.words("indonesian"))

def remove_stopwords(tokens):
    return [
        token for token in tokens
        if token not in stop_words
    ]

# Contoh
tokens = ["saya", "sedang", "belajar", "nlp", "yang", "menarik"]
print(remove_stopwords(tokens))

['belajar', 'nlp', 'menarik']


In [22]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

# Contoh
text = "Mahasiswa sedang mempelajari pemrograman."
print(stemmer.stem(text))

mahasiswa sedang ajar pemrograman


In [23]:
def preprocessing_pipeline(text):
    if not isinstance(text, str) or text.strip() == "":
        return []
    # 1. Cleaning
    text = clean_text(text)
    # 2. Tokenization
    tokens = word_tokenize(text)
    # 3. Stopword removal
    tokens = remove_stopwords(tokens)
    # 4. Stemming
    tokens = stem_tokens(tokens)
    return tokens

In [24]:
df_sample = df.head(5).copy()
df_sample["processed_tokens"] = df_sample[text_column].apply(preprocessing_pipeline)
df_sample["processed_text"] = df_sample["processed_tokens"].apply(lambda t: " ".join(t))

In [25]:
for i in range(5):
    print(f"RAW: {df_sample.iloc[i][text_column]}")
    print(f"PROCESSED: {df_sample.iloc[i]['processed_text']}")
    print("=" * 80)

RAW: Deretan Aktor yang Pernah Perankan Spider-Man
PROCESSED: deret aktor peran spiderman
RAW: Aktor Ryotaro Shimizu Ditemukan Meninggal Dunia di Kediamannya
PROCESSED: aktor ryotaro shimizu temu tinggal dunia diam
RAW: Aktor Marvelous Mrs. Maisel, Josh Grisetti Meninggal Dunia
PROCESSED: aktor marvelous mrs maisel josh grisetti tinggal dunia
RAW: Aktor Bittersweet Destiny, Wang Kai Ditemukan Meninggal Dunia di Rumahnya
PROCESSED: aktor bittersweet destiny wang kai temu tinggal dunia rumah
RAW: Aktor Thailand, Jorjay Phumhirun Mendadak Meninggal di Usia 20 Tahun
PROCESSED: aktor thailand jorjay phumhirun dadak tinggal usia


In [26]:
output_columns = [text_column, "clean_text", "processed_text"]

df[output_columns].to_csv(
    "scrape_kompas_clean.csv",
    index=False
)

print("Dataset berhasil disimpan.")

Dataset berhasil disimpan.
